# Operational Silver — Orders

## Mục tiêu hiện tại

Đọc và kiểm tra dữ liệu `orders` từ Bronze trước khi thiết kế tầng Silver.

Ở giai đoạn đầu, notebook chỉ thực hiện:

1. Xác nhận đường dẫn Bronze.
2. Xác nhận Databricks truy cập được dữ liệu Orders.
3. Kiểm tra định dạng vật lý của các file Bronze.
4. Chỉ sau khi đọc Bronze thành công mới khảo sát dữ liệu.

In [0]:
import os

account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
tenant_id = os.getenv("AZURE_TENANT_ID")
client_id = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")

endpoint = f"{account_name}.dfs.core.windows.net"

spark.conf.set(
    f"fs.azure.account.auth.type.{endpoint}",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{endpoint}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{endpoint}",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{endpoint}",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{endpoint}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
BRONZE_ROOT = (
    "abfss://bronze@fastorderdatalake.dfs.core.windows.net"
)
ORDERS_BRONZE_PATH = (
    f"{BRONZE_ROOT}/orders"
)

In [0]:
files = dbutils.fs.ls(
    ORDERS_BRONZE_PATH
)

display(files)

In [0]:
orders_date_path = files[0].path

print(
    "Orders ingestion date path:",
    orders_date_path
)

extraction_dirs = dbutils.fs.ls(
    orders_date_path
)

display(extraction_dirs)

In [0]:
first_extraction_path = (
    extraction_dirs[0].path
)

print(
    "First extraction:",
    first_extraction_path
)

batch_files = dbutils.fs.ls(
    first_extraction_path
)

display(batch_files)

In [0]:
parquet_path = (
    batch_files[0].path
)

print(
    "Parquet file:",
    parquet_path
)

In [0]:
binary_file = (
    spark.read
    .format("binaryFile")
    .load(parquet_path)
)

binary_row = (
    binary_file
    .select(
        "path",
        "length",
        "content",
    )
    .first()
)

print(
    "Path:",
    binary_row["path"]
)

print(
    "Dung lượng:",
    binary_row["length"],
    "bytes"
)

In [0]:
import pyarrow as pa
import pyarrow.parquet as pq


file_bytes = bytes(
    binary_row["content"]
)

buffer = pa.BufferReader(
    file_bytes
)

parquet_file = pq.ParquetFile(
    buffer
)

In [0]:
print(
    "     PARQUET PHYSICAL SCHEMA     "
)

print(
    parquet_file.schema
)

In [0]:
print(
    "    ARROW SCHEMA    "
)

print(
    parquet_file.schema_arrow
)

In [0]:
CUSTOMERS_BRONZE_PATH = (
    f"{BRONZE_ROOT}/customers"
)

In [0]:
customer_dates = dbutils.fs.ls(
    CUSTOMERS_BRONZE_PATH
)

customer_extractions = dbutils.fs.ls(
    customer_dates[0].path
)

customer_files = dbutils.fs.ls(
    customer_extractions[0].path
)

customer_parquet_path = (
    customer_files[0].path
)

print(
    customer_parquet_path
)

In [0]:
customer_binary_row = (
    spark.read
    .format("binaryFile")
    .load(customer_parquet_path)
    .select(
        "path",
        "length",
        "content",
    )
    .first()
)

print(
    "Path:",
    customer_binary_row["path"]
)

print(
    "Dung lượng:",
    customer_binary_row["length"],
    "bytes"
)

In [0]:
import pyarrow as pa
import pyarrow.parquet as pq

In [0]:
customer_buffer = (
    pa.BufferReader(
        bytes(
            customer_binary_row["content"]
        )
    )
)

customer_parquet_file = (
    pq.ParquetFile(
        customer_buffer
    )
)

In [0]:
print(
    "=== CUSTOMER ARROW SCHEMA ==="
)

print(
    customer_parquet_file.schema_arrow
)

In [0]:
print(
    "=== CUSTOMER PHYSICAL SCHEMA ==="
)

print(
    customer_parquet_file.schema
)